In [1]:
import os

# Disable TensorFlow completely for Hugging Face
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"


In [2]:
!pip install -U "transformers>=4.44.0" "peft>=0.11.0" "accelerate>=0.30.0" bitsandbytes scikit-learn openpyxl


Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python3 -m pip install --upgrade pip


In [3]:
pip install openpyxl

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
pip install peft

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
import numpy as np
import pandas as pd
import torch
import peft
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    BitsAndBytesConfig
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    TaskType,
)

# Fix randomness
def set_seed(seed: int = 42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [6]:
from sklearn.utils.class_weight import compute_class_weight
import torch.nn as nn

def get_weighted_loss(y_train, device):
    class_weights = compute_class_weight(
        class_weight="balanced",
        classes=np.unique(y_train),
        y=y_train
    )
    class_weights = torch.tensor(
        class_weights,
        dtype=torch.float,
        device=device
    )
    return nn.CrossEntropyLoss(weight=class_weights)


In [7]:
# ============================================================
# 2. Load dataset (same file used everywhere)
# ============================================================
file_path = "Reddit_Combined_Data.xlsx"  # adjust if needed
df = pd.read_excel(file_path, sheet_name="Train")

print("Loaded shape:", df.shape)
print("Columns:", df.columns.tolist())


Loaded shape: (800, 12)
Columns: ['score', 'selftext', 'subreddit', 'title', 'Root cause', 'Gender', 'Sentiment', 'Perceived severity', 'Perceived benefits', 'Perceived barriers', 'Cue to Action', 'Self-efficacy']


In [8]:
# ============================================================
# 3. Build text column (title + selftext)
# ============================================================
def combine_text(row):
    title = str(row.get("title", "") or "")
    body = str(row.get("selftext", "") or "")
    return title.strip() + " [SEP] " + body.strip()

df["text"] = df.apply(combine_text, axis=1)
print("Example combined text:")
print(df[["title", "selftext", "text"]].head(3))

Example combined text:
                                               title  \
0                        Do people get over anxiety?   
1  does anyone else have this big fear of suddenl...   
2         3 hour long panic attack after trying weed   

                                            selftext  \
0  Tried to watch this documentary “anxious Ameri...   
1  i’m currently laying in bed wide awake, feelin...   
2  Second time trying weed. First time felt close...   

                                                text  
0  Do people get over anxiety? [SEP] Tried to wat...  
1  does anyone else have this big fear of suddenl...  
2  3 hour long panic attack after trying weed [SE...  


In [9]:
# ============================================================
# 4. Basic cleaning for key raw columns
# ============================================================
def clean_str(x):
    if x is None:
        return np.nan
    x = str(x).strip()
    if x.lower() in ["nan", "none", "null", "", "na"]:
        return np.nan
    return x

for col in ["Perceived severity", "Self-efficacy", "Cue to Action", "Sentiment"]:
    if col in df.columns:
        df[col] = df[col].apply(clean_str)


In [10]:
# ============================================================
# 5. PERCEIVED SEVERITY  → severity_merged
# ============================================================
def map_severity(raw):
    if pd.isna(raw):
        return "Not mentioned"

    s = str(raw).lower()

    if "anxiety" in s or "panic" in s:
        return "Anxiety/Panic"
    if "depress" in s:
        return "Depression"
    if "ptsd" in s or "trauma" in s or "severe" in s:
        return "Severe/PTSD"
    if "lonely" in s or "isolation" in s or "alone" in s:
        return "Loneliness/Isolation"
    if "addict" in s or "alcohol" in s or "drug" in s or "substance" in s:
        return "Substance-related"

    return "Other"

df["severity_merged"] = df["Perceived severity"].apply(map_severity)
print("severity_merged distribution:")
print(df["severity_merged"].value_counts(dropna=False))

severity_merged distribution:
severity_merged
Depression           343
Anxiety/Panic        277
Other                138
Substance-related     23
Severe/PTSD           19
Name: count, dtype: int64


In [11]:
# ============================================================
# 6. SELF-EFFICACY (binary merged)  → Self_eff_merged
# ============================================================
def map_self_efficacy_binary(raw):
    if pd.isna(raw):
        return np.nan

    s = str(raw).strip()

    if s in ["Empowered Mindset", "Overcome Mindset"]:
        return "High self-efficacy"
    if s in ["Troubled Mindset", "Denial Mindset"]:
        return "Low self-efficacy"

    return np.nan

df["Self_eff_merged"] = df["Self-efficacy"].apply(map_self_efficacy_binary)
print("Self_eff_merged distribution:")
print(df["Self_eff_merged"].value_counts(dropna=False))

Self_eff_merged distribution:
Self_eff_merged
Low self-efficacy     519
High self-efficacy    281
Name: count, dtype: int64


In [12]:
# ============================================================
# 7. SELF-EFFICACY (4-way UNGROUPED) → df_selfeff4
# ============================================================
valid_self_eff_4 = [
    "Troubled Mindset",
    "Overcome Mindset",
    "Empowered Mindset",
    "Denial Mindset",
]

df_selfeff4 = df[df["Self-efficacy"].isin(valid_self_eff_4)].copy()
df_selfeff4["text_len"] = df_selfeff4["text"].astype(str).str.len()
df_selfeff4 = df_selfeff4[df_selfeff4["text_len"] >= 50].copy()

print("Self-efficacy 4-way distribution (filtered):")
print(df_selfeff4["Self-efficacy"].value_counts(dropna=False))
print("Rows in 4-way self-efficacy:", len(df_selfeff4))

Self-efficacy 4-way distribution (filtered):
Self-efficacy
Troubled Mindset     492
Overcome Mindset     150
Empowered Mindset    130
Denial Mindset        27
Name: count, dtype: int64
Rows in 4-way self-efficacy: 799


In [13]:
# ============================================================
# 8. CUE TO ACTION  → Cue_group
# ============================================================
def map_cue_action(raw):
    if pd.isna(raw):
        return "No clear action"

    s = str(raw).lower()

    if "help" in s or "therapy" in s or "treatment" in s or "doctor" in s:
        return "Help-seeking/treatment"
    if "advice" in s or "info" in s or "information" in s or "suggest" in s:
        return "Information seeking"
    if "share" in s or "experience" in s or "story" in s or "discussion" in s:
        return "Sharing/community"

    return "No clear action"

df["Cue_group"] = df["Cue to Action"].apply(map_cue_action)
print("Cue_group distribution:")
print(df["Cue_group"].value_counts(dropna=False))

Cue_group distribution:
Cue_group
No clear action           608
Information seeking       105
Help-seeking/treatment     87
Name: count, dtype: int64


In [14]:
# ============================================================
# 9. ROOT CAUSE  → Root_cause_final
#     (adapted to match your clean updated 2 logic)
# ============================================================
candidate_root_cols = [c for c in df.columns if "root" in c.lower() or "label" in c.lower()]
print("Root-cause candidate columns:", candidate_root_cols)

# If this is wrong, manually set ROOT_COL to the correct column (e.g. "Root cause")
ROOT_COL = candidate_root_cols[0]
print("Using root cause column:", ROOT_COL)

df_root = df.dropna(subset=[ROOT_COL]).copy()
df_root[ROOT_COL] = df_root[ROOT_COL].astype(str).str.strip()

print("Root raw value counts:")
print(df_root[ROOT_COL].value_counts(dropna=False))

root_map = {
    "Personality": "Personality",
    "personality": "Personality",

    "Trauma and stress": "Trauma/Stress",
    "Trauma & Stress": "Trauma/Stress",
    "Stress": "Trauma/Stress",
    "Trauma": "Trauma/Stress",

    "Early life": "Early Life",
    "Early life experiences": "Early Life",

    "Drug and alcohol": "Substance Use",
    "Drugs and alcohol": "Substance Use",
    "Substance use": "Substance Use",
    "Alcohol": "Substance Use",
    "Addiction": "Substance Use",
}

def map_root_cause(x):
    x = str(x).strip()
    return root_map.get(x, x)

df_root["Root_cause_clean"] = df_root[ROOT_COL].apply(map_root_cause)
print("Root_cause_clean value counts:")
print(df_root["Root_cause_clean"].value_counts(dropna=False))

min_samples_root = 30
vc_root = df_root["Root_cause_clean"].value_counts()
valid_labels_root = vc_root[vc_root >= min_samples_root].index.tolist()
print("Root cause labels kept:", valid_labels_root)

def merge_rare_root(label):
    if label in valid_labels_root:
        return label
    return "Others"

df_root["Root_cause_final"] = df_root["Root_cause_clean"].apply(merge_rare_root)
print("Root_cause_final distribution:")
print(df_root["Root_cause_final"].value_counts(dropna=False))

Root-cause candidate columns: ['Root cause']
Using root cause column: Root cause
Root raw value counts:
Root cause
Drug and Alcohol     200
Early life           200
Personality          200
Trauma and Stress    200
Name: count, dtype: int64
Root_cause_clean value counts:
Root_cause_clean
Drug and Alcohol     200
Early Life           200
Personality          200
Trauma and Stress    200
Name: count, dtype: int64
Root cause labels kept: ['Drug and Alcohol', 'Early Life', 'Personality', 'Trauma and Stress']
Root_cause_final distribution:
Root_cause_final
Drug and Alcohol     200
Early Life           200
Personality          200
Trauma and Stress    200
Name: count, dtype: int64


In [15]:
# ============================================================
# MERGE Root_cause_final BACK INTO df
# ============================================================

df = df.merge(
    df_root[["text", "Root_cause_final"]],
    on="text",
    how="left"
)

print("After merging Root_cause_final:")
print(df["Root_cause_final"].value_counts(dropna=False))


After merging Root_cause_final:
Root_cause_final
Trauma and Stress    203
Personality          201
Early Life           200
Drug and Alcohol     200
Name: count, dtype: int64


In [16]:
# ============================================================
# 10. PERCEIVED BENEFITS  → benefit_final
# ============================================================
candidate_benefit_cols = [c for c in df.columns if "benefit" in c.lower()]
print("Benefit candidate columns:", candidate_benefit_cols)

BEN_COL = candidate_benefit_cols[0]  # change manually if needed
print("Using benefit column:", BEN_COL)

df_benefit = df.dropna(subset=[BEN_COL]).copy()
df_benefit[BEN_COL] = df_benefit[BEN_COL].astype(str).str.strip()

print("Benefit raw value counts:")
print(df_benefit[BEN_COL].value_counts(dropna=False))

benefit_map = {
    "support": "Support",
    "finding support": "Support",

    "feeling heard": "Validation",
    "validation": "Validation",
    "feel understood": "Validation",

    "treatment": "Treatment",
    "access to treatment": "Treatment",
    "therapy": "Treatment",

    "not mentioned": "Not Mentioned",
    "none": "Not Mentioned",
}

def map_benefit(x):
    x = str(x).strip().lower()
    return benefit_map.get(x, x.title())

df_benefit["benefit_clean"] = df_benefit[BEN_COL].apply(map_benefit)
print("benefit_clean distribution:")
print(df_benefit["benefit_clean"].value_counts(dropna=False))

min_samples_ben = 30
vc_ben = df_benefit["benefit_clean"].value_counts()
valid_labels_ben = vc_ben[vc_ben >= min_samples_ben].index.tolist()
print("Benefit labels kept:", valid_labels_ben)

def merge_rare_benefits(label):
    if label in valid_labels_ben:
        return label
    return "Others"

df_benefit["benefit_final"] = df_benefit["benefit_clean"].apply(merge_rare_benefits)
print("benefit_final distribution:")
print(df_benefit["benefit_final"].value_counts(dropna=False))

Benefit candidate columns: ['Perceived benefits']
Using benefit column: Perceived benefits
Benefit raw value counts:
Perceived benefits
Not Mentioned                   458
Finding support                 157
Getting access to treatment     124
Improving quality of life        18
Feeling heard and understood     17
Making progress                  16
Feeling relieved                 11
Do not want to bother others      3
Name: count, dtype: int64
benefit_clean distribution:
benefit_clean
Not Mentioned                   458
Support                         157
Getting Access To Treatment     124
Improving Quality Of Life        18
Feeling Heard And Understood     17
Making Progress                  16
Feeling Relieved                 11
Do Not Want To Bother Others      3
Name: count, dtype: int64
Benefit labels kept: ['Not Mentioned', 'Support', 'Getting Access To Treatment']
benefit_final distribution:
benefit_final
Not Mentioned                  458
Support                        157
G

In [17]:
# ============================================================
# MERGE benefit_final BACK INTO df
# ============================================================

df = df.merge(
    df_benefit[["text", "benefit_final"]],
    on="text",
    how="left"
)

print("After merging benefit_final:")
print(df["benefit_final"].value_counts(dropna=False))


After merging benefit_final:
benefit_final
Not Mentioned                  470
Support                        169
Getting Access To Treatment    124
Others                          65
Name: count, dtype: int64


In [18]:
TASKS = {
    "sentiment": "Sentiment",
    "severity": "severity_merged",
    "self_efficacy_binary": "Self_eff_merged",
    "self_efficacy_4way": "Self-efficacy",
    "cue_to_action": "Cue_group",
    "root_cause": "Root_cause_final",
    "perceived_benefits": "benefit_final"
}


In [19]:
TASK_LR = {
    "sentiment": 2e-5,
    "severity": 1e-5,
    "self_efficacy_binary": 2e-5,
    "self_efficacy_4way": 1e-5,
    "cue_to_action": 2e-5,
    "root_cause": 1e-5,
    "perceived_benefits": 1e-5
}


In [20]:
class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        return {
            "input_ids": enc["input_ids"].squeeze(),
            "attention_mask": enc["attention_mask"].squeeze(),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }


In [21]:
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

results = []

for task_name, label_col in TASKS.items():
    print(f"\n=== Running task: {task_name} ===")

    df_task = df.dropna(subset=[label_col]).copy()

    le = LabelEncoder()
    df_task["label_id"] = le.fit_transform(df_task[label_col])

    num_labels = len(le.classes_)
    print("Classes:", le.classes_)

    X_train, X_test, y_train, y_test = train_test_split(
        df_task["text"].tolist(),
        df_task["label_id"].tolist(),
        test_size=0.2,
        stratify=df_task["label_id"],
        random_state=42
    )

    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME,
        trust_remote_code=True
    )
    tokenizer.padding_side = "left"
    tokenizer.pad_token = tokenizer.eos_token

    train_ds = TextDataset(X_train, y_train, tokenizer)
    test_ds = TextDataset(X_test, y_test, tokenizer)

    train_loader = DataLoader(train_ds, batch_size=4, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=4)

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=num_labels,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True
    )
    model.config.pad_token_id = tokenizer.pad_token_id    

    model = prepare_model_for_kbit_training(model)

    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=["q_proj", "v_proj"],
        lora_dropout=0.05,
        task_type="SEQ_CLS"
    )

    model = get_peft_model(model, lora_config)
    model.train()

    lr = TASK_LR[task_name]
    optimizer = AdamW(model.parameters(), lr=5e-5)
    loss_fn = get_weighted_loss(y_train, model.device)


    # ---- TRAIN ----
    for epoch in range(5):
        total_loss = 0
        for batch in train_loader:
            batch = {k: v.to(model.device) for k, v in batch.items()}
            loss = model(**batch).loss
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            total_loss += loss.item()
        print(f"Epoch {epoch+1} | Loss: {total_loss/len(train_loader):.4f}")

    # ---- EVAL ----
    model.eval()
    preds, gold = [], []

    with torch.no_grad():
        for batch in test_loader:
            batch = {k: v.to(model.device) for k, v in batch.items()}
            logits = model(**batch).logits
            preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
            gold.extend(batch["labels"].cpu().numpy())

    acc = accuracy_score(gold, preds)
    macro_f1 = f1_score(gold, preds, average="macro")
    weighted_f1 = f1_score(gold, preds, average="weighted")

    results.append({
        "task": task_name,
        "accuracy": acc,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1
    })

    print(acc, macro_f1, weighted_f1)



=== Running task: sentiment ===
Classes: ['Negative' 'Neutral' 'Positive']


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Some weights of Qwen2ForSequenceClassification were not initialized from the model checkpoint at Qwen/Qwen2.5-7B-Instruct and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:600: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.4 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/torch/utils/checkpoint.py:295: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with torch.enable_grad(), device_autocast_ctx, torch.cpu

Epoch 1 | Loss: 3.1840
Epoch 2 | Loss: 1.6489
Epoch 3 | Loss: 0.5902
Epoch 4 | Loss: 0.1306
Epoch 5 | Loss: 0.0701
0.5481927710843374 0.5494208494208495 0.5460855003023678

=== Running task: perceived_benefits ===
Classes: ['Getting Access To Treatment' 'Not Mentioned' 'Others' 'Support']


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Some weights of Qwen2ForSequenceClassification were not initialized from the model checkpoint at Qwen/Qwen2.5-7B-Instruct and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:600: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.4 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/torch/utils/checkpoint.py:295: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with torch.enable_grad(), device_autocast_ctx, torch.cpu

Epoch 1 | Loss: 3.9135
Epoch 2 | Loss: 1.7781
Epoch 3 | Loss: 0.7702
Epoch 4 | Loss: 0.1931
Epoch 5 | Loss: 0.0393
0.5421686746987951 0.4122957112686972 0.5388810239796666


In [22]:
results_df = pd.DataFrame(results)
print(results_df)

results_df.to_excel("qwen_7b_qlora_results.xlsx", index=False)


                   task  accuracy  macro_f1  weighted_f1
0             sentiment  0.668675  0.493977     0.668440
1              severity  0.572289  0.426668     0.575420
2  self_efficacy_binary  0.710843  0.660358     0.702955
3    self_efficacy_4way  0.566265  0.314707     0.545516
4         cue_to_action  0.728916  0.463782     0.709824
5            root_cause  0.548193  0.549421     0.546086
6    perceived_benefits  0.542169  0.412296     0.538881
